# Corrective RAG powered by LangDB

#### Introduction

Corrective Retrieval Augmented Generation (CRAG) is an advanced method that enhances the accuracy and relevance of responses generated by AI systems through a combination of retrieval and generation techniques. This approach aims to improve the effectiveness of retrieval-augmented generation models by incorporating corrective mechanisms. These mechanisms ensure that the documents used for generating answers are highly relevant and contextually appropriate. The original paper detailing CRAG can be found [here](https://arxiv.org/pdf/2401.15884).

Below is an explanation of CRAG, including its components.

#### Create tables and insert data

For CRAG, we will use two data sources. The first data source will consist of articles by the most popular authors from Wikipedia.

In [1]:
CREATE TABLE pdf_links
(
    link `String`
) engine = MergeTree
order by link;

In [2]:
CREATE TABLE pdf_contents (
    id `Int64`,
    content `String`,
    link `String`
)
ENGINE = MergeTree
ORDER BY content;

In [3]:
INSERT INTO pdf_links VALUES
  ('https://en.wikipedia.org/api/rest_v1/page/html/William_Shakespeare'),
  ('https://en.wikipedia.org/api/rest_v1/page/html/Agatha_Christie'),
  ('https://en.wikipedia.org/api/rest_v1/page/html/Barbara_Cartland'),
  ('https://en.wikipedia.org/api/rest_v1/page/html/Danielle_Steel'),
  ('https://en.wikipedia.org/api/rest_v1/page/html/Harold_Robbins');

In [4]:
INSERT INTO pdf_contents(link, content)
SELECT * FROM (SELECT JSONExtractString(metadata, 'link'), arrayStringConcat(groupArray(content), '')
FROM load_html((SELECT link FROM pdf_links))
GROUP BY JSONExtractString(metadata, 'link'));


Second data source will popQA dataset. This dataset will be used as fallback option.

In [5]:
CREATE TABLE popqa
ENGINE = Memory AS
SELECT *
FROM url('https://raw.githubusercontent.com/AlexTMallen/adaptive-retrieval/main/data/popQA.tsv', TSV)

In [6]:
select * from popqa limit 5

,id,subj,prop,obj,subj_id,prop_id,obj_id,s_aliases,o_aliases,s_uri,o_uri,s_wiki_title,o_wiki_title,s_pop,o_pop,question,possible_answers
0,4222362,George Rankin,occupation,politician,1850297,22,2834605,"""[""""George James Rankin""""]""","""[""""political leader"""",""""political figure"""",""""polit."""",""""pol""""]""",http://www.wikidata.org/entity/Q5543720,http://www.wikidata.org/entity/Q82955,George Rankin,Politician,142,25692,What is George Rankin's occupation?,"""[""""politician"""", """"political leader"""", """"political figure"""", """"polit."""", """"pol""""]"""
1,4725190,John Mayne,occupation,journalist,2079053,22,663400,[],"""[""""journo"""",""""journalists""""]""",http://www.wikidata.org/entity/Q6247345,http://www.wikidata.org/entity/Q1930187,John Mayne,Journalist,236,24952,What is John Mayne's occupation?,"""[""""journalist"""", """"journo"""", """"journalists""""]"""
2,4382392,Henry Feilden,occupation,politician,1925450,22,2834605,"""[""""Henry Master Feilden""""]""","""[""""political leader"""",""""political figure"""",""""polit."""",""""pol""""]""",http://www.wikidata.org/entity/Q5725578,http://www.wikidata.org/entity/Q82955,Henry Feilden (Conservative politician),Politician,58,25692,What is Henry Feilden's occupation?,"""[""""politician"""", """"political leader"""", """"political figure"""", """"polit."""", """"pol""""]"""
3,4822110,Kathy Saltzman,occupation,politician,2122743,22,2834605,[],"""[""""political leader"""",""""political figure"""",""""polit."""",""""pol""""]""",http://www.wikidata.org/entity/Q6377295,http://www.wikidata.org/entity/Q82955,Kathy Saltzman,Politician,127,25692,What is Kathy Saltzman's occupation?,"""[""""politician"""", """"political leader"""", """"political figure"""", """"polit."""", """"pol""""]"""
4,4011112,Eleanor Davis,occupation,cartoonist,1752619,22,68412,"""[""""Eleanor McCutcheon Davis""""]""","""[""""graphic artist"""",""""animator"""",""""illustrator""""]""",http://www.wikidata.org/entity/Q5354261,http://www.wikidata.org/entity/Q1114448,Eleanor Davis,Cartoonist,317,9649,What is Eleanor Davis's occupation?,"""[""""cartoonist"""", """"graphic artist"""", """"animator"""", """"illustrator""""]"""


### Embeddings
Langdb supports convenient way to generate embeddings on the fly. You can use the built-in `embed()` function for development and testing. 

You can also use `OpenAI` or other providers.

In [7]:
CREATE TABLE embeddings_fallback_data (
  id `Int64`,
  embeddings `Array`(`Float32`),
  question `String`
) 
ENGINE = MergeTree
ORDER BY id;

#### Spawn task
Spawn tasks to periodically calculate and insert embeddings for new questions into the fallback data embeddings table every 5 seconds

In [8]:
SPAWN TASK generate_embeddings
    BEGIN
INSERT INTO embeddings_fallback_data
SELECT p.id, embed(question), p.question 
FROM popqa AS p 
LEFT JOIN embeddings_fallback_data AS pe ON p.id = pe.id
WHERE p.id != pe.id
ORDER BY p.id
LIMIT 10
END EVERY 5 second;

Spawned Task: generate_embeddings with id: e7e53c63-92b6-4b10-9e92-ccb8a94f3986

#### Retrieval evaluator 

In the CRAG implementation, all documents are evaluated to filter out only the relevant ones. This process involves creating a prompt, model, and endpoint that return only the filtered documents.

In [13]:
CREATE MODEL retrieval_evaluator (
    question,
    document
) ENGINE = OpenAI(api_key = 'sk-proj-xxx', model_name = 'gpt-3.5-turbo')
PROMPT ( system "
Given a question, does the following document have exact
information to answer the question?
Question: {{question}}
Document: {{document}}
Think Step by step, and answer with yes or no only");

In [14]:
CREATE ENDPOINT get_correct_documents(question String) AS 
 with tbl as (
    select id, content, retrieval_evaluator($question, content) answer from pdf_contents
)
select id, answer, content
from tbl where positionCaseInsensitive(answer, 'yes') > 0
with description = "Endpoint to get the relevant document to the question";

#### Fallback data source model and prompt

For the fallback data source, we are using the popQA dataset. Based on the question, we attempt to find an answer with a similarity score within a defined threshold. The similarity score is calculated using the cosine distance between the embeddings of questions in dataset and the embedding of the asked question. If a suitable answer is found, it can be returned to the user.

In [24]:
CREATE MODEL evaluator(
    question,
    answer,
    score
) ENGINE = OpenAI(api_key = 'sk-proj-xxx', model_name = 'gpt-3.5-turbo')
PROMPT (system "Given a question and possible answer respond if it is relevant or not. 
Answer 'yes' if the score is below 0.1. If score is not provided, return 'no'.
Please use 'yes' or 'no' as the answer. 
question: {{question}}
possible answer: {{answer}}
similarity score: {{score}}");

Model already present

In [30]:
CREATE ENDPOINT evaluator(question String, answer String, score String) AS
SELECT evaluator($question, $answer, $score);

In [18]:
CREATE ENDPOINT endpoint_fallback_data(question String "Query to search question") AS
WITH tbl AS (
  SELECT CAST(embed($question) AS `Array`(`Float32`)) AS question_embed
)
SELECT cosineDistance(embeddings, question_embed) AS similarity, p.id, p.possible_answers
FROM embeddings_fallback_data AS pe
JOIN popqa AS p ON p.id = pe.id
CROSS JOIN tbl
ORDER BY similarity 
LIMIT 10
WITH description = "Returns top 10 results for the given query with similarity, id, and possible_answers";

In [19]:
CREATE ENDPOINT retrieval_evaluator(question String, document String) AS
SELECT retrieval_evaluator($question, $document)
WITH description = "Endpoint for evaluating the relevancy of an question and document";

In [20]:
CREATE MODEL IF NOT EXISTS fallback_model (
  input
) ENGINE = OpenAI(api_key = 'sk-proj-xxx', model_name = 'gpt-3.5-turbo')
PROMPT (system "Given a question, find the document relevant to the question and return possible answers and similarity scores. 
Question: {{input}}")
TOOLS (endpoint_fallback_data)
SETTINGS retries = 1;

#### Main model

The main model attempts to find an answer in Wikipedia articles. It uses the get_correct_documents tool, which was designed to retrieve only relevant documents. If no relevant documents are found, the model then attempts to use the fallback data source.


In [26]:
CREATE MODEL main(
    question
) ENGINE = OpenAI(api_key = 'sk-proj-xxx', model_name = 'gpt-3.5-turbo')
PROMPT (system "Given a question, try to find a relevant answer based on relavent documents. 
Relevant documents can be fetched with get_correct_documents tool.
If you cannot get any document, try to use the fallback data source.
Fallback data source can only be used if the main data source doesn't provide a relevant answer.
You must verify the answer with the evaluator. If the evaluator returns 'no', do not provide an answer to the user.
question: {{question}}")
TOOLS (get_correct_documents, endpoint_fallback_data, evaluator)
SETTINGS retries = 1;

Model already present

In [31]:
SELECT main('Who written hamlet?')

,main('Who written hamlet?')
0,"The play ""Hamlet"" was written by William Shakespeare."
